# Vehicle Damage Type Classification — Colab Training

Trains the CNN damage-type classifier on a free Colab GPU, running the **same
`src/` code as the repo** (no duplicated training logic, so results are reproducible
locally later).

## One-time setup

1. In Google Drive, create a folder called **`cardd`** at the top level of *My Drive*.
2. Upload these two files into it (both were generated in your project root):
   - `cardd_classification.zip` — the 8,578-image dataset (378 MB)
   - `cardd_src.zip` — the repo source code (34 KB)
3. **Runtime → Change runtime type → T4 GPU**, then Save.
4. Run the cells top to bottom.

Only step 4 needs repeating in later sessions — the Drive uploads persist.

> Re-upload `cardd_src.zip` whenever you change anything under `src/`.

## 1. Verify the GPU is attached

In [ ]:
import torch

print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
else:
    print("NO GPU — go to Runtime > Change runtime type > T4 GPU, then re-run.")
    print("Training on CPU works but takes roughly 20-30x longer.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Settings

`config.py` in the repo defaults to `resnet18`, but this project is built around
**MobileNetV2**, so that is the default here. Change `BACKBONE` to compare
architectures — `resnet18`, `resnet50`, `mobilenet_v2` or `efficientnet_b0`.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/cardd'   # where you uploaded the two zips
PROJECT   = '/content/AI-Image-Processing'   # unpacked on Colab's fast local disk

BACKBONE    = 'mobilenet_v2'
EPOCHS      = 15
BATCH_SIZE  = 32
NUM_WORKERS = 2      # Linux can parallelise JPEG decode; Windows config uses 0

## 4. Unpack code and data

The dataset is extracted to Colab's **local disk**, not Drive. Training reads every
image each epoch, and Drive I/O is slow enough to leave the GPU idle most of the time.

In [ ]:
import os, zipfile, time
from pathlib import Path

for z in ('cardd_src.zip', 'cardd_classification.zip'):
    p = Path(DRIVE_DIR) / z
    assert p.exists(), f"Missing {p} — check the one-time setup steps above."

os.makedirs(PROJECT, exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/cardd_src.zip') as zf:
    zf.extractall(PROJECT)
print(f"code unpacked ({time.time() - t0:.1f}s)")

t0 = time.time()
os.makedirs(f'{PROJECT}/data', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE_DIR}/cardd_classification.zip') as zf:
    zf.extractall(f'{PROJECT}/data')
print(f"data unpacked ({time.time() - t0:.1f}s)")

## 5. Sanity-check the dataset

In [ ]:
from pathlib import Path

root = Path(PROJECT) / 'data' / 'classification'
total = 0
for split in ('train', 'val', 'test'):
    counts = {d.name: len(list(d.glob('*.jpg')))
              for d in sorted((root / split).iterdir()) if d.is_dir()}
    n = sum(counts.values())
    total += n
    print(f"{split:5s} {n:5d}   {counts}")
print(f"\ntotal {total} images (expected 8578)")

## 6. Dependencies

Colab already ships torch, torchvision, scikit-learn, matplotlib, seaborn and
pandas, so this usually installs nothing.

In [ ]:
import importlib.util

PIP_NAME = {'sklearn': 'scikit-learn', 'PIL': 'pillow', 'cv2': 'opencv-python'}
MODULES = ('torch', 'torchvision', 'sklearn', 'matplotlib', 'seaborn',
           'numpy', 'pandas', 'PIL', 'cv2', 'tqdm')

missing = [m for m in MODULES if importlib.util.find_spec(m) is None]
names = ' '.join(PIP_NAME.get(m, m) for m in missing)
print("installing:", names if names else "(nothing — all dependencies present)")

In [ ]:
if names:
    !pip install -q {names}

## 7. Train

Runs `src/train_cnn.py` unchanged. It saves the checkpoint with the best
**validation macro-F1**, so the epoch marked `<- best (saved)` is the one kept —
that is not necessarily the last epoch.

In [ ]:
import os

os.chdir(PROJECT)
!python src/train_cnn.py --backbone {BACKBONE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

## 8. Training curves

In [ ]:
from IPython.display import Image, display

display(Image(f'{PROJECT}/results/cnn_training_history.png'))

## 9. Evaluate on the held-out test set

Produces accuracy, per-class precision / recall / F1, and the confusion matrix.

In [ ]:
import os

os.chdir(PROJECT)
!python src/evaluate.py

## 10. Confusion matrix

In [ ]:
from IPython.display import Image, display

display(Image(f'{PROJECT}/results/confusion_matrix.png'))

## 11. Save results back to Drive

Colab wipes local disk when the session ends. This copies the trained weights and
every result file into `MyDrive/cardd/results_<backbone>/` so they survive.

In [ ]:
import shutil
from pathlib import Path

dest = Path(DRIVE_DIR) / f'results_{BACKBONE}'
dest.mkdir(parents=True, exist_ok=True)

for f in sorted(Path(f'{PROJECT}/results').glob('*')):
    shutil.copy2(f, dest / f.name)
    print("saved", f.name)

ckpt = Path(f'{PROJECT}/models/cnn_damage_classifier.pth')
if ckpt.exists():
    shutil.copy2(ckpt, dest / ckpt.name)
    print(f"saved {ckpt.name} ({ckpt.stat().st_size / 1e6:.1f} MB)")

print(f"\nEverything is in Drive at: {dest}")

## Bringing the results home

Download `results_<backbone>/` from Drive and drop the contents into your local repo:

- `cnn_damage_classifier.pth` → `models/`
- everything else → `results/`

Both folders are gitignored, so nothing large lands in git. Your local
`src/predict.py` and `src/evaluate.py` will then work against the trained weights —
`evaluate.py` needs torch locally, but `results/classification_report.txt` and
`confusion_matrix.png` already contain the numbers for the report.